# Agent memory

Build agents with **long-term memory** using Azure AI Foundry's Memory API.

## What you'll learn

| Scenario | Description |
|----------|-------------|
| **1. Memory Store** | Create a store backed by local models |
| **2. Store Memories** | Extract memories from conversations |
| **3. Scope Isolation** | Keep user data separate |
| **4. Agent + Memory** | Agent with `memory_search` tool |
| **5. Cross-Session** | Memory persists across sessions |

## Theme: space exploration expert

This lab uses a **space exploration** theme - the agent remembers users' favourite planets, space interests, and exploration preferences.

## Prerequisites

1. **Python environment**: Run `uv sync` from the repository root to create the
   shared `.venv`, then select the `.venv` kernel in VS Code.
2. **`.env` file**: Must be populated by the `04-foundry-project-pattern-setup` labs.
   No additional `.env` keys are required beyond the standard setup.
3. **Azure CLI**: Run `az login` before executing the cells.
4. **Permissions**: Your identity needs **Owner** or **Contributor** + **User Access Administrator**
   on the new `rg-foundry-memory-{suffix}` resource group to create RBAC assignments.

## Key constraint: Memory API requires local model deployments

The Memory API's `memory_search` tool does not support BYO gateway models. Agents using
`memory_search` must reference a model deployed directly on the same Foundry account - the
`core-alpha/gpt-4.1-mini` connection format is not supported for this tool. This lab deploys
a dedicated Foundry account (`aif-memory-{suffix}`) with local model deployments into a
separate resource group (`rg-foundry-memory-{suffix}`) that is excluded from the
`deny-model-deployments` policy applied to spoke resource groups.

## Step 1: Configuration

In [1]:
import os, subprocess, hashlib, json, base64, time
from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display, Markdown, clear_output

repo_root = Path(subprocess.run(
    'git rev-parse --show-toplevel', shell=True, capture_output=True, text=True
).stdout.strip())
load_dotenv(repo_root / '.env', override=True)

# Suffix - first 6 chars of SHA-256 of subscription ID
SUB_ID          = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
SUBSCRIPTION_ID = SUB_ID
SUFFIX          = hashlib.sha256((SUB_ID + 'v2').encode()).hexdigest()[:6]
TEAM_NAME       = "alpha"
LOCATION        = "eastus2"

# Dedicated memory resource group - separate from rg-foundry-spoke-alpha-{suffix} so the
# deny-model-deployments policy does not block the local model deployments needed here
MEMORY_RG = f"rg-foundry-memory-{SUFFIX}"

# Principal ID from JWT token (avoids graph.microsoft.com network call - same pattern as the project spoke and multi-project deployments)
token        = subprocess.run('az account get-access-token --query accessToken -o tsv', shell=True, capture_output=True, text=True).stdout.strip()
payload      = token.split('.')[1] + '=='
PRINCIPAL_ID = json.loads(base64.b64decode(payload))['oid']

# Memory API models - deployed locally on aif-memory-{suffix} (not routed via APIM)
LOCAL_CHAT_MODEL  = "gpt-4.1-mini"
EMBEDDING_MODEL   = "text-embedding-3-small"
MEMORY_STORE_NAME = "space-expert-memory"

print(f"Suffix:           {SUFFIX}")
print(f"Memory RG:        {MEMORY_RG}")
print(f"Local chat model: {LOCAL_CHAT_MODEL}")
print(f"Embedding model:  {EMBEDDING_MODEL}")

Suffix:           c2676f
Memory RG:        rg-foundry-memory-c2676f
Local chat model: gpt-4.1-mini
Embedding model:  text-embedding-3-small


## Step 2: Create resource group

Creates a dedicated `rg-foundry-memory-{suffix}` resource group. This is kept separate from
`rg-foundry-spoke-alpha-{suffix}` because the `deny-model-deployments` policy is assigned
to that RG - local model deployments required by the Memory API would be blocked there.

In [2]:
!az group create -n "{MEMORY_RG}" -l "{LOCATION}" -o table

Location    Name
----------  ------------------------
eastus2     rg-foundry-memory-c2676f


## Step 3: Deploy memory infrastructure

Deploys into `rg-foundry-memory-{suffix}`. Resources created:

| Resource | Name | Purpose |
|----------|------|---------|
| Foundry Account | `aif-memory-{suffix}` | Hosts local model deployments for Memory API |
| `gpt-4.1-mini` deployment | `gpt-4.1-mini` | Summarisation and fact extraction (Memory API internal) |
| `text-embedding-3-small` deployment | `text-embedding-3-small` | Semantic indexing and search (Memory API internal) |
| Foundry Project | `project-alpha-memory-{suffix}` | Memory store and agent workspace |

> Takes approximately 4-5 minutes.

In [3]:
!az deployment group create -g "{MEMORY_RG}" --template-file main.bicep \
    -p suffix="{SUFFIX}" \
    -p teamName="{TEAM_NAME}" \
    -p location="{LOCATION}" \
    -p deployerPrincipalId="{PRINCIPAL_ID}" \
    -p localChatModel="{LOCAL_CHAT_MODEL}" \
    -p embeddingModelName="{EMBEDDING_MODEL}" \
    -o table

=A new Bicep release is available: v0.43.8. Upgrade now by running "az bicep upgrade".
Name    State      Timestamp                         Mode         ResourceGroup
------  ---------  --------------------------------  -----------  ------------------------
main    Succeeded  2026-05-10T15:27:18.097604+00:00  Incremental  rg-foundry-memory-c2676f


## Step 4: Get deployment outputs

In [4]:
r = subprocess.run(
    f'az deployment group show -g "{MEMORY_RG}" -n main --query properties.outputs -o json',
    shell=True, capture_output=True, text=True
)
outputs = json.loads(r.stdout)

ACCOUNT_NAME     = outputs['accountName']['value']
PROJECT_NAME     = outputs['projectName']['value']
PROJECT_ENDPOINT = outputs['projectEndpoint']['value']
LOCAL_CHAT       = outputs['localChatModel']['value']
EMBEDDING        = outputs['embeddingModelName']['value']

print(f"Account:          {ACCOUNT_NAME}")
print(f"Project:          {PROJECT_NAME}")
print(f"Project Endpoint: {PROJECT_ENDPOINT}")
print(f"Local Chat:       {LOCAL_CHAT}")
print(f"Embedding:        {EMBEDDING}")

# Persist outputs to .env under ALPHA_MEMORY_* keys
env_file = repo_root / '.env'
existing = {}
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if '=' in line and not line.startswith('#'):
            k, _, v = line.partition('=')
            existing[k.strip()] = v.strip()

existing.update({
    'ALPHA_MEMORY_FOUNDRY_ACCOUNT':  ACCOUNT_NAME,
    'ALPHA_MEMORY_PROJECT_ENDPOINT': PROJECT_ENDPOINT,
})
env_file.write_text('\n'.join(f'{k}={v}' for k, v in existing.items()) + '\n')
print(f"\nOutputs saved to .env under ALPHA_MEMORY_* keys")

Account:          aif-memory-c2676f
Project:          project-alpha-memory-c2676f
Project Endpoint: https://aif-memory-c2676f.services.ai.azure.com/api/projects/project-alpha-memory-c2676f
Local Chat:       gpt-4.1-mini
Embedding:        text-embedding-3-small

Outputs saved to .env under ALPHA_MEMORY_* keys


## Step 5: Wait for RBAC propagation

The project managed identity needs Foundry User and Cognitive Services OpenAI User roles on
the account before the Memory API will accept requests. Azure RBAC assignments typically
propagate within 60 seconds but can take up to a few minutes.

In [5]:
import time
from IPython.display import clear_output

for i in range(60, 0, -10):
    clear_output(wait=True)
    print(f"⏳ RBAC propagation... {i}s")
    time.sleep(10)
clear_output(wait=True)
print("✅ Ready")

✅ Ready


## Step 6: Setup project client

In [6]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client = project_client.get_openai_client()

print(f"✅ Project client ready: {PROJECT_ENDPOINT}")

✅ Project client ready: https://aif-memory-c2676f.services.ai.azure.com/api/projects/project-alpha-memory-c2676f


## Step 7: Setup memory client

`MemoryClient` wraps the Memory REST API. It uses the `https://ai.azure.com` token audience
(distinct from the standard management plane) and API version `2025-11-15-preview`.

In [7]:
from memory_helpers import MemoryClient, build_conversation
from display_helpers import show_store_created, show_memories, show_search_results, show_agent_created, show_conversation, show_error

memory = MemoryClient(ACCOUNT_NAME, PROJECT_NAME)
print(f"✅ Memory client ready")

✅ Memory client ready


---
## Scenario 1: Create memory store

The memory store uses **local models** for internal processing.

In [8]:
result = memory.create_store(
    name=MEMORY_STORE_NAME,
    chat_model=LOCAL_CHAT,
    embedding_model=EMBEDDING,
    description="Space exploration preferences and conversation history",
    user_profile_details="Favorite planets, space missions, exploration interests, celestial phenomena preferences"
)

if 'error' not in result:
    show_store_created(MEMORY_STORE_NAME, LOCAL_CHAT, EMBEDDING)
else:
    show_error(result['error'])

### Memory Store Created

Property,Value
Name,space-expert-memory
Chat Model,gpt-4.1-mini
Embedding Model,text-embedding-3-small
Status,✅ Created


---
## Scenario 2: Store user memories

Extract and store memories from conversations using the Memory API.

In [9]:
# Fix import and define test users
from IPython.display import display, Markdown

USER_ALICE = "user_alice_123"
USER_BOB = "user_bob_456"

display(Markdown('''
| User | Scope ID | Profile |
|------|----------|---------|
| Alice | `user_alice_123` | Loves Mars, interested in rover missions, wants to see the northern lights |
| Bob | `user_bob_456` | Saturn fan, fascinated by rings and moons, dreams of Europa exploration |
'''))


| User | Scope ID | Profile |
|------|----------|---------|
| Alice | `user_alice_123` | Loves Mars, interested in rover missions, wants to see the northern lights |
| Bob | `user_bob_456` | Saturn fan, fascinated by rings and moons, dreams of Europa exploration |


In [10]:
# Store Alice's preferences
alice_msgs = build_conversation(
    "Mars is my absolute favorite planet! I'm fascinated by the Perseverance rover and Ingenuity helicopter missions. I also really want to see the northern lights on Earth someday - they're on my bucket list.",
    "Got it! Mars is your favorite, you love the rover missions, and you're dreaming of seeing the aurora borealis. I'll remember that!"
)

print("⏳ Processing Alice's memories...")
result = memory.update_memories(MEMORY_STORE_NAME, USER_ALICE, alice_msgs)

if 'error' not in result:
    show_memories("Alice's Memories Stored", result.get('memories', []))
else:
    show_error(result['error'])

⏳ Processing Alice's memories...
✅ Alice's Memories Stored - No new memories extracted


In [11]:
# Store Bob's preferences
bob_msgs = build_conversation(
    "Saturn is definitely my favorite - those rings are just spectacular! I'm really interested in its moon Europa and the possibility of life in its subsurface ocean. I also love following the James Webb telescope discoveries.",
    "Saturn fan with a love for those iconic rings! You're curious about Europa's ocean and following JWST discoveries. Got it!"
)

print("⏳ Processing Bob's memories...")
result = memory.update_memories(MEMORY_STORE_NAME, USER_BOB, bob_msgs)

if 'error' not in result:
    show_memories("Bob's Memories Stored", result.get('memories', []))
else:
    show_error(result['error'])

⏳ Processing Bob's memories...
✅ Bob's Memories Stored - No new memories extracted


---
## Scenario 3: Search memories (scope isolation)

Verify each user only sees their own memories.

In [12]:
query = "Which planet should I learn more about?"
display(Markdown(f'**Query:** "{query}"'))

# Each user only sees their own memories
alice_result = memory.search_memories(MEMORY_STORE_NAME, USER_ALICE, query)
bob_result = memory.search_memories(MEMORY_STORE_NAME, USER_BOB, query)

show_search_results("Alice", "👩", alice_result.get('memories', []))
show_search_results("Bob", "👨", bob_result.get('memories', []))

display(Markdown('✅ **Scope isolation verified** - each user sees only their own memories'))

**Query:** "Which planet should I learn more about?"

#### 👩 Alice's Memories

Type,Content
user_profile,User wants to witness the northern lights (aurora borealis) on Earth and has this as a bucket list i...
user_profile,User's favorite planet is Mars.
user_profile,User is fascinated by the Perseverance rover and Ingenuity helicopter missions on Mars.


#### 👨 Bob's Memories

Type,Content
user_profile,User is a fan of Saturn and its rings.
user_profile,User has a strong interest in Europa and the potential for life in its subsurface ocean.
user_profile,User follows discoveries from the James Webb Space Telescope (JWST).


✅ **Scope isolation verified** - each user sees only their own memories

---
## Scenario 4: Agent with memory

Create an agent that uses `memory_search` tool.

> ⚠️ **Current Limitation**: The `memory_search` tool is **not supported with BYO (gateway) models**.
> Error: `"The following tools are not supported with BYO model: memory_search. Please remove these tools or use a standard model deployment."`
> 
> **Workaround**: Use a local model deployment for agents with memory tools.
> Once this limitation is lifted, you can switch back to gateway models (`connection/model` format).

In [13]:
from azure.ai.projects.models import PromptAgentDefinition

AGENT_NAME = "SpaceExpert"

def create_agent_for_user(scope: str) -> tuple:
    """Create an agent scoped to a specific user."""
    agent = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=LOCAL_CHAT,
            instructions="You are a friendly space exploration expert. Personalize recommendations based on user's favorite planets and space interests. Remember their specific interests in missions, phenomena, and celestial bodies. Always use the memory tool before giving an answer.",
            tools=[{
                "type": "memory_search",
                "memory_store_name": MEMORY_STORE_NAME,
                "scope": scope,
                "update_delay": 1
            }]
        )
    )
    return agent

# Create agents for each user
agent_alice = create_agent_for_user(USER_ALICE)
agent_bob = create_agent_for_user(USER_BOB)

display(Markdown('''
### Agents Created
| User | Agent Version | Memory Scope |
|------|--------------|--------------|
| Alice | `''' + agent_alice.version + '''` | `user_alice_123` |
| Bob | `''' + agent_bob.version + '''` | `user_bob_456` |

> ⚠️ Using local model (gateway not supported with `memory_search`)
'''))


### Agents Created
| User | Agent Version | Memory Scope |
|------|--------------|--------------|
| Alice | `1` | `user_alice_123` |
| Bob | `2` | `user_bob_456` |

> ⚠️ Using local model (gateway not supported with `memory_search`)


In [14]:
query = "Hi! I want to learn something fascinating about space today. What would you recommend based on my interests?"
display(Markdown(f'**Query:** "{query}"'))

# Alice's recommendation
response_alice = openai_client.responses.create(
    input=query,
    extra_body={"agent_reference": {"name": agent_alice.name, "version": agent_alice.version, "type": "agent_reference"}}
)
alice_response = response_alice.output_text if hasattr(response_alice, 'output_text') else str(response_alice.output)

# Bob's recommendation
response_bob = openai_client.responses.create(
    input=query,
    extra_body={"agent_reference": {"name": agent_bob.name, "version": agent_bob.version, "type": "agent_reference"}}
)
bob_response = response_bob.output_text if hasattr(response_bob, 'output_text') else str(response_bob.output)

display(Markdown('---'))
show_conversation("👩 Alice's Recommendation", query, alice_response)
display(Markdown('---'))
show_conversation("👨 Bob's Recommendation", query, bob_response)

display(Markdown('''
### ✅ Same query, different answers!
- **Alice** gets Mars/rover mission recommendations (loves Mars and Perseverance)
- **Bob** gets Saturn/Europa recommendations (fascinated by rings and subsurface oceans)
'''))

**Query:** "Hi! I want to learn something fascinating about space today. What would you recommend based on my interests?"

---

### 👩 Alice's Recommendation

Role,Message
👤 User,Hi! I want to learn something fascinating about space today. What would you recommend based on my interests?
🤖 Agent,"Since you're fascinated by Mars, especially the Perseverance rover and Ingenuity helicopter missions, here's something fascinating: Did you know that Perseverance not only explores the Martian surface but also carries an instrument called MOXIE (Mars Oxygen In-Situ Resource Utilization Experiment) that can produce oxygen from Mars' carbon dioxide atmosphere? This is a major step forward for future human missions, as it could allow astronauts to generate breathable oxygen and rocket fuel right on Mars. And tying this to your interest in celestial phenomena like the northern lights on Earth, Mars has its version of auroras too! While ours are caused by Earth's magnetic field interacting with solar particles, Mars' auroras are quite different because it lacks a global magnetic field. Instead, localized magnetic patches in the crust create beautiful but more scattered auroras. Imagine witnessing a Martian light show—much rarer and more mysterious than what we see here. Would you like me to share more about these Martian auroras or details on how Ingenuity manages powered flight in Mars’ thin atmosphere?"


---

### 👨 Bob's Recommendation

Role,Message
👤 User,Hi! I want to learn something fascinating about space today. What would you recommend based on my interests?
🤖 Agent,"Hi! Since you're fascinated by Saturn, especially its rings, and deeply interested in Europa and the potential for life in its subsurface ocean, plus keeping up-to-date with JWST discoveries, here's something that might captivate you: Did you know that recent observations from the James Webb Space Telescope have started to shed light on the complex chemistry of icy moons like Europa? JWST's powerful infrared instruments can detect subtle signatures of organic molecules and water vapor plumes that might be escaping Europa’s ice shell. This complements what we know about Saturn’s rings, which are made mostly of ice particles but also hint at dynamic processes around Saturn. Moreover, new studies suggest that Saturn's rings might act like a time machine, preserving clues about the age and evolution of the Saturn system, including the tidal forces that heat Europa’s ocean beneath its icy crust—creating conditions that could foster life. If you want, I can dive deeper into the latest JWST findings related to Europa or share amazing facts about Saturn’s rings and what they reveal about planetary science. Which would you like to explore first?"



### ✅ Same query, different answers!
- **Alice** gets Mars/rover mission recommendations (loves Mars and Perseverance)
- **Bob** gets Saturn/Europa recommendations (fascinated by rings and subsurface oceans)


---
## Scenario 4b: Single agent version with `{{$userId}}` scope

The per-version-per-user pattern above doesn't scale. The Memory API supports one alternative:
set `scope="{{$userId}}"` in the agent definition. The service resolves this server-side to
`{tid}_{oid}` from the caller's Entra auth token on every request - no extra_body or
per-call override needed.

> **Production behaviour:** each end-user authenticates with their own Entra token →
> their requests automatically land in an isolated scope.
>
> **This lab:** `DefaultAzureCredential` resolves to one developer identity, so all
> calls share the same scope. The single-version pattern is real; the isolation is not
> demonstrable here without per-user tokens.

In [15]:
# One agent version for all users - scope resolved per-request from the caller's Entra token
agent_shared = project_client.agents.create_version(
    agent_name="SpaceExpertShared",
    definition=PromptAgentDefinition(
        model=LOCAL_CHAT,
        instructions="You are a friendly space exploration expert. Personalize recommendations based on the user's interests. Always use the memory tool before answering.",
        tools=[{
            "type": "memory_search",
            "memory_store_name": MEMORY_STORE_NAME,
            "scope": "{{$userId}}",  # resolved server-side from caller's Entra TID+OID
            "update_delay": 1
        }]
    )
)

response = openai_client.responses.create(
    input="What space topics have I been interested in?",
    extra_body={"agent_reference": {"name": agent_shared.name, "version": agent_shared.version, "type": "agent_reference"}}
)

display(Markdown(f"**Agent version:** `{agent_shared.version}` (single version, shared by all users)"))
display(Markdown(f"**Scope in definition:** `{{{{$userId}}}}` → resolved to `{{tid}}_{{oid}}` of caller"))
show_conversation("SpaceExpertShared Response", "What space topics have I been interested in?", response.output_text)

**Agent version:** `1` (single version, shared by all users)

**Scope in definition:** `{{$userId}}` → resolved to `{tid}_{oid}` of caller

### SpaceExpertShared Response

Role,Message
👤 User,What space topics have I been interested in?
🤖 Agent,"I currently don't have any record of your specific interests in space topics. Could you tell me a bit about what excites you in space exploration? For example, are you fascinated by planets, black holes, human spaceflight, or something else?"


---
## Scenario 5: Automatic memory extraction

Demonstrate that the agent **automatically learns** from conversations - no manual `update_memories()` needed!

> 📝 **How it works:**
> - The `memory_search` tool has `update_delay` set (we use 1 second for demo)
> - After each response, the system automatically extracts memories
> - Chat summaries are enabled in our memory store (`chat_summary_enabled: True`)

In [16]:
USER_CHARLIE = "user_charlie_789"
agent_charlie = create_agent_for_user(USER_CHARLIE)

In [17]:
display(Markdown('### Turn 1: Charlie chats with the agent'))

charlie_msg1 = "Hi! I'm really excited about the upcoming solar eclipse next month. I want to find the best viewing spot and learn about what causes them."

response1 = openai_client.responses.create(
    input=charlie_msg1,
    extra_body={"agent_reference": {"name": agent_charlie.name, "version": agent_charlie.version, "type": "agent_reference"}}
)
charlie_response1 = response1.output_text if hasattr(response1, 'output_text') else str(response1.output)

show_conversation("Charlie's First Message", charlie_msg1, charlie_response1, "Charlie")

# Continue the conversation
charlie_msg2 = "That sounds great! By the way, Jupiter is my favorite planet. I love learning about the Great Red Spot and the Galilean moons."

response2 = openai_client.responses.create(
    input=charlie_msg2,
    extra_body={"agent_reference": {"name": agent_charlie.name, "version": agent_charlie.version, "type": "agent_reference"}}
)
charlie_response2 = response2.output_text if hasattr(response2, 'output_text') else str(response2.output)

display(Markdown('---'))
show_conversation("Charlie's Second Message", charlie_msg2, charlie_response2, "Charlie")

### Turn 1: Charlie chats with the agent

### Charlie's First Message

Role,Message
👤 Charlie,Hi! I'm really excited about the upcoming solar eclipse next month. I want to find the best viewing spot and learn about what causes them.
🤖 Agent,"Hi there! Solar eclipses are truly spectacular events. To help you find the best viewing spot, could you please tell me where you'll be located or where you plan to travel for the eclipse? Also, if you'd like, I can explain the fascinating science behind what causes solar eclipses."


---

### Charlie's Second Message

Role,Message
👤 Charlie,"That sounds great! By the way, Jupiter is my favorite planet. I love learning about the Great Red Spot and the Galilean moons."
🤖 Agent,"Jupiter is an amazing choice! The Great Red Spot is such a fascinating storm—it's been raging for centuries and is actually larger than Earth. The Galilean moons—Io, Europa, Ganymede, and Callisto—are incredible too, each with unique features. For example, Europa is thought to have a subsurface ocean that might harbor life. If you're interested, I can share more about upcoming missions like NASA’s Europa Clipper, which aims to explore Europa’s habitability, or dive deeper into the dynamics of the Great Red Spot and how it changes over time. Would you like to hear more about any of these?"


In [18]:
# Wait for automatic memory extraction
display(Markdown('### ⏳ Waiting for automatic memory extraction...'))
display(Markdown('> The `memory_search` tool automatically extracts and stores memories after `update_delay` seconds of inactivity.'))

import time
for i in range(30, 0, -10):
    clear_output(wait=True)
    display(Markdown(f'### ⏳ Waiting for memory extraction... {i}s'))
    time.sleep(10)

clear_output(wait=True)
print("✅ Memory extraction should be complete")

✅ Memory extraction should be complete


In [19]:
# Turn 2: New conversation - ask what we talked about
display(Markdown('### Turn 2: New conversation - test if agent remembers'))

charlie_msg3 = "What have we recently been talking about?"

response3 = openai_client.responses.create(
    input=charlie_msg3,
    extra_body={"agent_reference": {"name": agent_charlie.name, "version": agent_charlie.version, "type": "agent_reference"}}
)
charlie_response3 = response3.output_text if hasattr(response3, 'output_text') else str(response3.output)

show_conversation("Charlie Asks About Previous Chat", charlie_msg3, charlie_response3, "Charlie")

display(Markdown('''
### ✅ Automatic Memory Extraction Works!

**What just happened:**
1. Charlie chatted about solar eclipses & Jupiter being their favorite planet
2. We did **NOT** call `update_memories()` manually
3. The `memory_search` tool automatically extracted and stored the conversation
4. In a new conversation, the agent remembers what we discussed!

**This is the magic of the `memory_search` tool** - it handles extraction automatically.
'''))

### Turn 2: New conversation - test if agent remembers

### Charlie Asks About Previous Chat

Role,Message
👤 Charlie,What have we recently been talking about?
🤖 Agent,"Recently, we've been talking about your favorite planet Jupiter, focusing on fascinating features like the Great Red Spot and the Galilean moons (Io, Europa, Ganymede, and Callisto). We also discussed the upcoming solar eclipse in June 2026, including your interest in the best places to view it and the science behind solar eclipses. Would you like to dive deeper into any of these topics or explore something new related to space?"



### ✅ Automatic Memory Extraction Works!

**What just happened:**
1. Charlie chatted about solar eclipses & Jupiter being their favorite planet
2. We did **NOT** call `update_memories()` manually
3. The `memory_search` tool automatically extracted and stored the conversation
4. In a new conversation, the agent remembers what we discussed!

**This is the magic of the `memory_search` tool** - it handles extraction automatically.


---
## Summary

## Key learnings

| Concept | Detail |
|---------|--------|
| Memory API Models | Must be deployed **locally** (not via gateway) |
| Agent with `memory_search` | Also requires local model |
| Token Audience | `https://ai.azure.com` |
| Responses API | `openai_client.responses.create()` with `agent_reference` |

## Current limitation

> ⚠️ **`memory_search` tool does not support BYO (gateway) models**
> 
> Error: `"The following tools are not supported with BYO model: memory_search"`

## Files

| File | Purpose |
|------|---------|
| `memory_helpers.py` | `MemoryClient` class, `build_conversation()` |
| `display_helpers.py` | Display functions for tables and results |
| `main.bicep` | Infrastructure (local model deployments + Foundry project) |

In [20]:
# Uncomment to delete all memory infrastructure.
# The spoke and core resource groups (the core gateway and project spoke deployments) are not affected.
# !az group delete -n "{MEMORY_RG}" --yes --no-wait
# print(f"Deleting {MEMORY_RG}")